# 01 · Left sources: Oracle, files and credentials

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_left/backends/test_sources.py
just test oracle        # while it runs: docker ps  -> gvenzl/oracle-free, the on-prem stand-in
```
**Read:** `src/dtrack_left/backends/sources.py` (`SqlSource.partition_counts`, `pull_window`,
`OracleSource._pick`, `FileSource.relation`) and `src/dtrack_left/common/dates.py`.

In [ ]:
import os

import nbkit
import pandas as pd
from nbkit import code, show, table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1 · Credentials resolve by name

A pair only names its connection (`conn_macro`). `OracleSource` resolves each field in order:
explicit field → `*_env` variable → `<macro>_USR/_PWD/_DSN` → service (`service` field or
`DTRACK_ORACLE_MACROS`, through LDAP when `LDAP_DSN` is set, resolved at connect time) →
global `ORACLE_*`. A macro with a service is never overridden by the global `ORACLE_DSN`.

In [ ]:
from dtrack_left.backends.sources import OracleSource

for name in ("LDAP_DSN", "PCDS_USR"):
    os.environ.pop(name, None)
os.environ |= {
    "pb30_USR": "u30",
    "pb30_PWD": "p30",
    "DTRACK_ORACLE_MACROS": "pb30:svc30",
    "ORACLE_DSN": "global",
}
base = {"source": "oracle", "table": "ORDERS", "date_col": "BUSINESS_DATE"}
rows = []
for macro in ("pb30", "other"):
    source = OracleSource({**base, "conn_macro": macro})
    rows.append(
        {
            "conn_macro": macro,
            "user": source.user,
            "service": source.service,
            "dsn used": source.resolve_dsn(),
        }
    )
table(pd.DataFrame(rows))

## 2 · Why `col-check` must resolve credentials again

`row-check` writes the pair config to S3 with every secret removed; `conn_macro` is only a name,
so it survives. `col-check` does not reuse row-check's connection: it rebuilds the source from
the manifest's `left` block for every window, so the name is all it has.

In [ ]:
from dtrack_left.checks.row_check import SECRET_FIELDS, public_pair_config

pair = {
    "left": {
        **base,
        "conn_macro": "pb30",
        "user": "u30",
        "password": "p30",
        "dsn": "h:1521/S",
        "user_env": "X",
    },
    "right": {"source": "databricks", "table": "orders", "date_col": "business_date"},
    "col_map": {"AMOUNT": "amount"},
}
print("removed before S3:", ", ".join(sorted(SECRET_FIELDS)))
show(public_pair_config(pair)["left"], "manifest left block (what col-check receives)")

**About the fields in this demo.** The pair above is deliberately over-filled to show what gets
stripped. In a real config, `conn_macro` alone is enough:

| Field | Meaning | Keep it in a real config? |
| --- | --- | --- |
| `conn_macro` | a name; credentials resolve from `<macro>_USR`, `<macro>_PWD`, `<macro>_DSN` or the service map | **yes**, the only field that works in both stages |
| `user`, `password` | literal credentials; highest priority | no: secrets in a file, and removed before S3 |
| `dsn` | a literal connection string; beats every lookup | no: removed before S3, so `row-check` would use it and `col-check` would not |
| `user_env` (and `password_env`, `dsn_env`) | the *name* of a variable to read, for names that break the `<macro>_*` convention | only locally; also removed before S3 |

The trap: an inline `dsn` or `*_env` works in `row-check` but is gone by `col-check`, which
rebuilds the source from the stripped manifest. With several Oracle servers, use `conn_macro`
plus `<macro>_*` variables in `.env`.

## 3 · Files become SQL relations through DuckDB

`FileSource` turns csv / txt / json / ndjson / parquet into a relation, so counting and pulling
reuse exactly the SQL path of Oracle. Text formats are read with `all_varchar = true`.

In [ ]:
from dtrack_left.backends.sources import FileSource

folder = nbkit.sandbox("01")
(folder / "orders.txt").write_text(
    "D|ID|AMOUNT\n2026-01-01|007|1.50\n2026-01-01|7|2.0\n2026-01-02|8|20260927\n"
)
source = FileSource(
    {"path": str(folder / "orders.txt"), "date_col": "D", "options": {"delim": "|"}}
)
code(source.reader().replace(str(folder), "<folder>"), "sql", "DuckDB reader")
print("partition counts:", source.partition_counts())
code(
    source.pull_window(
        ["2026-01-01", "2026-01-02"], ["AMOUNT"], folder / "raw.csv", key_columns=["ID"]
    ).read_text(),
    label="pulled window, all_varchar = true",
)

**Why `all_varchar`:** with type inference DuckDB rewrites values before we ever see them.
`007` and `7` may be different business keys; `1.50` must reach the statistics as written so
both sides apply the *same* normalisation at compare time.

In [ ]:
typed = FileSource(
    {
        "path": str(folder / "orders.txt"),
        "date_col": "D",
        "options": {"delim": "|", "all_varchar": False},
    }
)
code(
    typed.pull_window(
        ["2026-01-01", "2026-01-02"], ["AMOUNT"], folder / "typed.csv", key_columns=["ID"]
    ).read_text(),
    label="pulled window, all_varchar = false",
)

## 4 · Patching odd files with `{src}` and `date_format`

The pair's `query` wraps the reader (`{src}`), e.g. to derive the partition from the file name;
`date_format` parses the date column and switches the predicate to real dates.

In [ ]:
for day in ("20260101", "20260102"):
    (folder / f"orders_{day}.csv").write_text("ID,V\n1,a\n2,b\n")
patched = FileSource(
    {
        "path": str(folder / "orders_*.csv"),
        "options": {"filename": True},
        "query": "SELECT *, regexp_extract(filename, '(\\d{8})', 1) AS D FROM {src}",
        "date_col": "D",
        "date_format": "%Y%m%d",
    }
)
code(patched.relation().replace(str(folder), "<folder>"), "sql", "patched relation")
print("partition counts:", patched.partition_counts())

## 5 · The SQL every source generates

Only two statements per source: a `GROUP BY` count and a window pull. A hole in the window
(a partition that did not match in Stage 1) splits the predicate into contiguous ranges.

In [ ]:
from dtrack_left.common.dates import contiguous_runs

calls = []
oracle = OracleSource({**base, "date_type": "date", "user": "u", "password": "p", "dsn": "d"})
oracle.connect = lambda: (
    calls.append(nbkit.support.FakeConnection(lambda sql: (["D", "N"], []))) or calls[-1]
)
oracle.partition_counts("2026-01-01", "2026-01-31")
days = [f"2026-01-{d:02d}" for d in range(1, 31) if d != 15]
oracle.pull_window(days, ["AMOUNT"], folder / "w.csv", key_columns=["ORDER_ID"])
for label, connection in zip(("partition_counts", "pull_window"), calls):
    code(connection.statements[0], "sql", label)
print("contiguous runs:", contiguous_runs(days))

## Check yourself

<details><summary>pb30 has a service in DTRACK_ORACLE_MACROS and ORACLE_DSN=global is set. Which dsn?</summary>

`svc30`. A macro with a service ignores the global DSN; only a macro with no DSN and no service
falls back to `ORACLE_DSN`.
</details>

<details><summary>Why does the window predicate use two BETWEEN ranges instead of one?</summary>

The missing day (01-15) did not match in Stage 1. `contiguous_runs` splits the admitted days so
the excluded partition is never read, and range predicates stay index-friendly on Oracle.
</details>